In [1]:
# Ray Data Fair Scheduler — Shared Pool Starvation Repro
#
# Pipeline:
#   instant_source (0s) -> SlowActor (15s) -> SlowerActor (10s) -> fast_sink (0s)
#
# Issue:
#   instant_source finishes immediately. All 32 items queue for SlowActor at once.
#   Autoscaler ramps SlowActor every few ms (util=2.0 >= 1.75) before it produces
#   any output. After ~12s the shared CPU pool is fully exhausted by SlowActor.
#   SlowerActor starts with 0 input until t~15s. By then SlowActor holds all shared
#   CPUs, and SlowerActor is capped at its reserved allocation only.
#
#   Even though SlowerActor is the true bottleneck (highest per-actor latency when
#   capped at 4 actors), util >= 1.75 passes Gate 1 but Gate 3 blocks scale-up:
#   budget = reserved = 4 CPUs  =>  3x throughput loss on the critical path.
#
# Budget (16 CPUs, reservation_ratio=0.5, 2 eligible ActorPool ops):
#   reserved/op = 4  |  shared = 8
#   SlowActor steals shared when it exceeds 4 actors (reserved)
#   At 12 actors: shared fully exhausted -> SlowerActor capped at 4

import time, threading, collections
import ray, ray.data
from ray.data import ActorPoolStrategy

print(f"Ray version: {ray.__version__}")


Ray version: 2.56.0


In [2]:
NUM_FILES    = 32
SLOW_DELAY   = 15.0  # SlowActor: 15s/item; with 32 tasks, util=32/12=2.67>=1.75 all the way to budget cap
SLOWER_DELAY = 10.0  # SlowerActor: true bottleneck, starved at reserved CPUs only

# Budget (16 CPUs, ratio=0.5, 2 ops):
RESERVED   = 4   # CPUs per op
SHARED     = 8   # shared pool
STARVED_AT = RESERVED + SHARED  # slow actors needed to exhaust shared = 12

print(f"SlowerActor if NOT starved: {STARVED_AT} actors / {SLOWER_DELAY:.0f}s = {STARVED_AT/SLOWER_DELAY:.2f} files/sec")
print(f"SlowerActor when starved  : {RESERVED} actors / {SLOWER_DELAY:.0f}s = {RESERVED/SLOWER_DELAY:.2f} files/sec")
print(f"=> {STARVED_AT//RESERVED}x throughput loss on the critical path")


SlowerActor if NOT starved: 12 actors / 10s = 1.20 files/sec
SlowerActor when starved  : 4 actors / 10s = 0.40 files/sec
=> 3x throughput loss on the critical path


In [3]:
import ray as _ray

@_ray.remote(num_cpus=0)
class _Counter:
    def __init__(self):
        self._alloc       = collections.defaultdict(int)
        self._active      = collections.defaultdict(int)
        self._peak_alloc  = collections.defaultdict(int)
        self._peak_active = collections.defaultdict(int)
    def alloc(self, s):
        self._alloc[s] += 1
        self._peak_alloc[s] = max(self._peak_alloc[s], self._alloc[s])
    def dealloc(self, s):
        self._alloc[s] = max(0, self._alloc[s] - 1)
    def enter(self, s):
        self._active[s] += 1
        self._peak_active[s] = max(self._peak_active[s], self._active[s])
    def exit(self, s):
        self._active[s] = max(0, self._active[s] - 1)
    def snapshot(self):
        return {"alloc": dict(self._alloc), "active": dict(self._active),
                "peak_alloc": dict(self._peak_alloc), "peak_active": dict(self._peak_active)}

def instant_source(batch):
    return {"item_id": list(range(NUM_FILES))}

def fast_sink(batch):
    return batch

class SlowActor:
    def __init__(self):
        _ray.get_actor("_ctr").alloc.remote("slow")
    def __del__(self):
        try: _ray.get_actor("_ctr").dealloc.remote("slow")
        except Exception: pass
    def __call__(self, batch):
        ctr = _ray.get_actor("_ctr")
        ctr.enter.remote("slow")
        try:
            time.sleep(SLOW_DELAY)
            return batch
        finally:
            ctr.exit.remote("slow")

class SlowerActor:
    def __init__(self):
        _ray.get_actor("_ctr").alloc.remote("slower")
    def __del__(self):
        try: _ray.get_actor("_ctr").dealloc.remote("slower")
        except Exception: pass
    def __call__(self, batch):
        ctr = _ray.get_actor("_ctr")
        ctr.enter.remote("slower")
        try:
            time.sleep(SLOWER_DELAY)
            return batch
        finally:
            ctr.exit.remote("slower")


In [4]:
ray.shutdown()
ray.init(num_cpus=16)

ctr = _Counter.options(name="_ctr").remote()
snapshots = []
_stop = threading.Event()

def _monitor():
    t0 = time.perf_counter()
    total = ray.cluster_resources().get("CPU", 16)
    while not _stop.is_set():
        snap = ray.get(ctr.snapshot.remote())
        snapshots.append((
            time.perf_counter() - t0,
            snap["alloc"],
            snap["active"],
            total - ray.available_resources().get("CPU", total),
        ))
        _stop.wait(1.0)

t0 = time.perf_counter()
ds = (
    ray.data.from_items([{"seed": 0}])
    .map_batches(instant_source, batch_size=1)
    .repartition(NUM_FILES)
    .map_batches(SlowActor,   batch_size=1, num_cpus=1,
                 compute=ActorPoolStrategy(min_size=1, max_size=NUM_FILES))
    .map_batches(SlowerActor, batch_size=1, num_cpus=1,
                 compute=ActorPoolStrategy(min_size=1, max_size=NUM_FILES))
    .map_batches(fast_sink, batch_size=1, num_cpus=0)
)

monitor = threading.Thread(target=_monitor, daemon=True)
monitor.start()
results = ds.take_all()
_stop.set()
monitor.join(timeout=3)

peaks = ray.get(ctr.snapshot.remote())
print(f"Done: {len(results)} items in {time.perf_counter()-t0:.1f}s")
print(f"Peak actors — slow: {peaks['peak_alloc'].get('slow',0)}  slower: {peaks['peak_alloc'].get('slower',0)}")


2026-07-14 21:51:40,178	INFO worker.py:2015 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8269 


2026-07-14 21:51:42,573	INFO streaming_executor.py:193 -- Starting execution of Dataset dataset_5_0. Full logs are in /tmp/ray/session_2026-07-14_21-51-26_677860_2186124/logs/ray-data


2026-07-14 21:51:42,575	INFO streaming_executor.py:194 -- Execution plan of Dataset dataset_5_0: InputDataBuffer[Input] -> TaskPoolMapOperator[MapBatches(instant_source)] -> AllToAllOperator[Repartition] -> ActorPoolMapOperator[MapBatches(SlowActor)] -> ActorPoolMapOperator[MapBatches(SlowerActor)] -> TaskPoolMapOperator[MapBatches(fast_sink)]


[2026-07-14 21:51:42,617 E 2186124 2186124] core_worker.cc:2149: Actor with class name: 'MapWorker(MapBatches(SlowActor))' and ID: 'd714c01c220de29caff2d43501000000' has constructor arguments in the object store and max_restarts > 0. If the arguments in the object store go out of scope or are lost, the actor restart will fail. See https://github.com/ray-project/ray/issues/53727 for more details.
2026-07-14 21:51:42,643	INFO __init__.py:56 -- Progress will be logged because stdout is a non-interactive terminal.


2026-07-14 21:51:42,682	WARNING resource_manager.py:766 -- Cluster resources are not enough to run any task from TaskPoolMapOperator[MapBatches(instant_source)]. The job may hang forever unless the cluster scales up.


...

2026-07-14 21:52:53,362	INFO logging_progress.py:233 --   Tasks: 0; Actors: 0; Queued blocks: 0 (0.0B); Resources: 0.0 CPU, 8.0B object store


2026-07-14 21:52:53,363	INFO logging_progress.py:192 -- ============================================


(raylet) [2026-07-14 21:52:55,240 E 2187411 2187442] (raylet) file_system_monitor.cc:116: /tmp/ray/session_2026-07-14_21-51-26_677860_2186124 is over 95% full, available space: 0.0260582 GB; capacity: 8 GB. Object creation will fail if spilling is required.


2026-07-14 21:52:58,244	INFO streaming_executor.py:327 -- ✔️  Dataset dataset_5_0 execution finished in 75.67 seconds


Done: 32 items in 76.5s
Peak actors — slow: 12  slower: 11


In [5]:
print("=== Timeline (1s samples) ===")
print(f"  {'t':>5}  {'slow_alloc':>10} {'slow_active':>11}  "
      f"{'slower_alloc':>12} {'slower_active':>13}  {'cpu':>4}  note")
print("  " + "-"*90)

slower_started = False
for ts, alloc, active, cpu in snapshots:
    sl_a  = alloc.get("slow",   0)
    sl_t  = active.get("slow",  0)
    sr_a  = alloc.get("slower", 0)
    sr_t  = active.get("slower",0)

    stolen     = max(0, sl_a - RESERVED)
    rem_shared = max(0, SHARED - stolen)
    sr_budget  = RESERVED + rem_shared

    note = ""
    if sr_a == 0 and sl_a > 0:
        note = f"<- slower: 0 input  (slow stealing shared, slower budget if started={sr_budget})"
    elif not slower_started and sr_t > 0:
        slower_started = True
        note = f"<- slower FIRST task  slow={sl_a} actors, shared stolen={stolen}, slower budget={sr_budget}"
    elif sl_a >= STARVED_AT and sr_a <= RESERVED:
        note = f"<- STARVED: slower capped at reserved={RESERVED}, util>=1.75 but Gate 3 blocks"

    print(f"  {ts:5.1f}s  {sl_a:>10d} {sl_t:>11d}  "
          f"{sr_a:>12d} {sr_t:>13d}  {cpu:>3.0f}  {note}")

print()

# Summary
peak_slow = max((s[1].get("slow", 0) for s in snapshots), default=0)
sr_starved = [s[1].get("slower", 0) for s in snapshots if s[1].get("slow", 0) >= STARVED_AT]
peak_sr_starved = max(sr_starved, default=0)

print(f"SlowActor   peak: {peak_slow} actors  (exhausts shared pool at >= {STARVED_AT})")
print(f"SlowerActor during starvation: {peak_sr_starved} actors  "
      f"(reserved={RESERVED}, potential={STARVED_AT})")
potential_tput = STARVED_AT / SLOWER_DELAY
actual_tput    = max(1, peak_sr_starved) / SLOWER_DELAY
print(f"Throughput loss on critical path: "
      f"{potential_tput:.2f} vs {actual_tput:.2f} files/sec"
      f"  => {int(round(potential_tput / actual_tput))}x slower")


=== Timeline (1s samples) ===
      t  slow_alloc slow_active  slower_alloc slower_active   cpu  note
  ------------------------------------------------------------------------------------------
    0.2s           0           0             0             0    2  
    1.2s           1           0             1             0    0  
    2.2s           1           1             1             0    3  
    3.2s           2           2             1             0    4  
    4.2s           3           3             1             0    5  
    5.2s           5           4             1             0    6  
    6.2s           5           5             1             0    7  
    7.2s           6           6             1             0    8  
    8.3s           6           6             1             0    8  
    9.3s           7           7             1             0    9  
   10.3s           7           7             1             0   10  
   11.3s           8           8             1           